# OCR Pipeline with Azure + Supabase
Notebook này:
- Load PaddleOCR-VL model đúng **1 lần**
- Duyệt qua danh sách `video_id` từ CSV
- Mỗi video: download keyframe từ Azure → chạy OCR → upload JSON lên Azure → cập nhật Supabase
- Cleanup local sau mỗi video để tiết kiệm disk

In [1]:
# !git clone https://github.com/CallmeAndree/NLP4B/ -b itshoang2024/dev

In [2]:
from pathlib import Path

# Lấy thư mục chứa notebook này
NOTEBOOK_DIR = Path().resolve()  # hoặc Path(__file__).parent nếu dùng script
REPO_ROOT = NOTEBOOK_DIR.parent.parent
print(REPO_ROOT)

D:\Code\HCMUS\NLP4B\NLP4B\data-processing


In [3]:
import os
import sys
import shutil

import pandas as pd
from azure.storage.blob import BlobServiceClient, ContentSettings
from supabase import create_client
from dotenv import load_dotenv
load_dotenv()


# ===== Secrets =====
os.environ["AZURE_STORAGE_CONNECTION_STRING"] = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
os.environ["SUPABASE_URL"] = os.getenv("SUPABASE_URL")
os.environ["SUPABASE_KEY"] = os.getenv("SUPABASE_KEY")

# ===== Runtime config =====
RUNNER = "all"
START_ID, END_ID = 0, 2000

OCR_MODEL_ID = "PaddlePaddle/PaddleOCR-VL-1.5"
OCR_REVISION = "main"
TORCH_DTYPE = "auto"     # auto → float16 on CUDA, float32 on CPU
BATCH_SIZE = 6
SAVE_EVERY = 50
MAX_NEW_TOKENS = 512

# ===== HF cache config =====
HF_CACHE_ROOT = REPO_ROOT / "data" / "hf_home"
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# Ví dụ: "/kaggle/input/paddleocr-cache/hf_home"
HF_DATASET_CACHE_ROOT = None

# ===== Paths =====
OCR_SRC_PATH = REPO_ROOT / "src" / "ocr"
OUTPUT_OCR_DIR = Path(REPO_ROOT / "data" / "ocr")
LOCAL_KEYFRAME_ROOT = Path(REPO_ROOT / "data" / "keyframes")
OUTPUT_OCR_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_KEYFRAME_ROOT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(OCR_SRC_PATH))

# ===== External services =====
supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])
blob_service_client = BlobServiceClient.from_connection_string(os.environ["AZURE_STORAGE_CONNECTION_STRING"], logging_enable=False)
ocr_container_client = blob_service_client.get_container_client("ocr")
try:
    ocr_container_client.get_container_properties()
except Exception:
    blob_service_client.create_container("ocr")

df = pd.read_csv(REPO_ROOT / "data" / "video_id" / f"{RUNNER}.csv")
print(f"Loaded {len(df)} assigned videos for runner={RUNNER}")

c:\Users\Hoang\miniconda3\envs\agentic_env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Loaded 1122 assigned videos for runner=all


## HF Cache (optional)

Nếu đã có Kaggle Dataset chứa model cache, cell này sẽ copy để tránh download lại.

In [4]:
from huggingface_hub import snapshot_download

def ensure_ocr_cache(
    model_id: str,
    revision: str,
    cache_root: Path,
    dataset_cache_root: str | None = None,
) -> Path:
    cache_root = Path(cache_root)

    # 1) Ưu tiên dùng cache từ Kaggle Dataset
    if dataset_cache_root:
        dataset_cache_root = Path(dataset_cache_root)
        if dataset_cache_root.exists():
            print(f"[cache] Copying HF cache from Kaggle Dataset: {dataset_cache_root}")
            if cache_root.exists():
                shutil.rmtree(cache_root)
            shutil.copytree(dataset_cache_root, cache_root)
            print(f"[cache] HF cache copied to: {cache_root}")

            model_dirs = list(cache_root.glob("**/snapshots/*"))
            if model_dirs:
                local_model_dir = model_dirs[0]
                print(f"[cache] Using cached model dir: {local_model_dir}")
                return local_model_dir

            print("[cache] Copied dataset cache but no snapshot dir found. Will fallback to download.")
        else:
            print(f"[cache] Dataset cache path not found, fallback to download: {dataset_cache_root}")

    # 2) Download nếu chưa có
    print(f"[cache] snapshot_download model={model_id} revision={revision}")
    local_model_dir = snapshot_download(
        repo_id=model_id,
        revision=revision,
        cache_dir=str(cache_root),
    )
    print(f"[cache] Model cached at: {local_model_dir}")
    return Path(local_model_dir)

LOCAL_OCR_MODEL_DIR = ensure_ocr_cache(
    model_id=OCR_MODEL_ID,
    revision=OCR_REVISION,
    cache_root=HF_CACHE_ROOT,
    dataset_cache_root=HF_DATASET_CACHE_ROOT,
)
print("LOCAL_OCR_MODEL_DIR =", LOCAL_OCR_MODEL_DIR)

[cache] snapshot_download model=PaddlePaddle/PaddleOCR-VL-1.5 revision=main


c:\Users\Hoang\miniconda3\envs\agentic_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 19 files: 100%|██████████| 19/19 [00:00<?, ?it/s]

[cache] Model cached at: D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--PaddlePaddle--PaddleOCR-VL-1.5\snapshots\6819afc8509ac9afa50e91b34627a7cf8f7900bb
LOCAL_OCR_MODEL_DIR = D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--PaddlePaddle--PaddleOCR-VL-1.5\snapshots\6819afc8509ac9afa50e91b34627a7cf8f7900bb


## Hàm tải keyframe từ Azure

In [5]:
KEYFRAME_CONTAINER = "keyframes"

def download_video_keyframes(video_id: str, local_root: str | Path = LOCAL_KEYFRAME_ROOT) -> str:
    container_client = blob_service_client.get_container_client(KEYFRAME_CONTAINER)
    local_dir = Path(local_root) / video_id
    local_dir.mkdir(parents=True, exist_ok=True)

    blobs = list(container_client.list_blobs(name_starts_with=f"{video_id}/"))
    if not blobs:
        raise FileNotFoundError(f"No keyframes found on Azure for video_id={video_id}")

    for blob in blobs:
        filename = Path(blob.name).name
        local_path = local_dir / filename
        with open(local_path, "wb") as f:
            data = container_client.download_blob(blob.name).readall()
            f.write(data)

    return str(local_dir)

## Load PaddleOCR-VL model đúng 1 lần

Dùng `paddle_ocr.load_model()` để load model 1 lần duy nhất, tái sử dụng `runtime` cho toàn bộ batch.

In [6]:
import paddle_ocr as ocr

runtime = ocr.load_model(
    model_id=str(LOCAL_OCR_MODEL_DIR),
    hf_cache_dir=None,               # đã resolve cache ở cell trước
    revision=OCR_REVISION,
    device="cuda" if __import__("torch").cuda.is_available() else "cpu",
    torch_dtype=TORCH_DTYPE,
)

print("Runtime ready. Model source:", runtime["model_source"])

[2026-04-15 22:13:27] [ocr] INFO     Before model load | cuda_available=False | ram_used_pct=54.9 | ram_available_gb=12.55
[2026-04-15 22:13:27] [ocr] INFO     Using local model path: D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--PaddlePaddle--PaddleOCR-VL-1.5\snapshots\6819afc8509ac9afa50e91b34627a7cf8f7900bb
[2026-04-15 22:13:27] [ocr] INFO     Loading PaddleOCR-VL config from: D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--PaddlePaddle--PaddleOCR-VL-1.5\snapshots\6819afc8509ac9afa50e91b34627a7cf8f7900bb
c:\Users\Hoang\miniconda3\envs\agentic_env\Lib\site-packages\transformers\modeling_rope_utils.py:927: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the mea

Runtime ready. Model source: D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--PaddlePaddle--PaddleOCR-VL-1.5\snapshots\6819afc8509ac9afa50e91b34627a7cf8f7900bb


## Loop OCR cho nhiều video_id

Pipeline mỗi video:
1. Download keyframes từ Azure
2. Chạy OCR bằng `paddle_ocr.process_directory()`
3. Upload `<video_id>_ocr.json` lên Azure container `ocr`
4. Cập nhật Supabase `video_processing_progress.ocr = True`
5. Cleanup local files

In [7]:
processed_videos = []
failed_videos = []

for i in range(START_ID, min(END_ID, len(df))):
    video_id_with_ext = df.iloc[i]["video_id"]
    video_id = str(video_id_with_ext).replace(".mp4", "")
    print(f"\n[{i}] Processing video: {video_id}")

    local_input_dir = None
    ocr_output_file = None

    try:
        # A. Download keyframes từ Azure
        local_input_dir = download_video_keyframes(video_id)

        # B. Run OCR bằng runtime đã load sẵn
        ocr_output_file = ocr.process_directory(
            input_dir=local_input_dir,
            output_dir=OUTPUT_OCR_DIR,
            runtime=runtime,
            batch_size=BATCH_SIZE,
            save_every=SAVE_EVERY,
            max_new_tokens=MAX_NEW_TOKENS,
        )

        # C. Upload JSON lên Azure
        print(f"[{i}] Uploading OCR to Azure...")
        blob_name = f"{video_id}/{ocr_output_file.name}"
        blob_client = ocr_container_client.get_blob_client(blob_name)
        with open(ocr_output_file, "rb") as data:
            blob_client.upload_blob(
                data,
                overwrite=True,
                content_settings=ContentSettings(content_type="application/json"),
            )

        # D. Update Supabase progress
        print(f"[{i}] Updating DB progress...")
        supabase.table("video_processing_progress").update({"ocr": True}).eq("video_id", video_id).execute()

        processed_videos.append(video_id)

    except Exception as e:
        print(f"[{i}] FAILED: {video_id} | error={e}")
        failed_videos.append({"index": i, "video_id": video_id, "error": str(e)})

    finally:
        # E. Cleanup local artifacts theo từng video
        if ocr_output_file and Path(ocr_output_file).exists():
            Path(ocr_output_file).unlink()

        if local_input_dir and Path(local_input_dir).exists():
            shutil.rmtree(local_input_dir)

        ocr.cleanup_memory()

print("\nDone.")
print("Processed videos:", processed_videos)
print("Failed videos:", failed_videos)


[0] Processing video: w2nmzwknVco


[2026-04-15 22:13:39] [ocr] INFO     Video w2nmzwknVco | total_images=45 | already_processed=0 | remaining=45
OCR w2nmzwknVco:  12%|█▎        | 1/8 [04:04<28:32, 244.71s/batch]


KeyboardInterrupt: 